Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(14)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [15]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 5)
Las dimensiones de testX son:  (10529, 12, 5)
Las dimensiones de valX son:  (5186, 12, 5)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 22s - 77ms/step - ia: 0.2228 - loss: 1.1432 - mae: 0.8834 - rmse: 1.0675 - smape: 1.5343 - val_ia: 0.2102 - val_loss: 0.8748 - val_mae: 0.7789 - val_rmse: 0.9275 - val_smape: 1.8272

Epoch 2/128                                           

287/287 - 6s - 20ms/step - ia: 0.1836 - loss: 1.0721 - mae: 0.8580 - rmse: 1.0341 - smape: 1.5959 - val_ia: 0.2126 - val_loss: 0.8646 - val_mae: 0.7741 - val_rmse: 0.9226 - val_smape: 1.9133

Epoch 3/128                                           

287/287 - 10s - 36ms/step - ia: 0.1634 - loss: 1.0470 - mae: 0.8484 - rmse: 1.0221 - smape: 1.6285 - val_ia: 0.2137 - val_loss: 0.8614 - val_mae: 0.7725 - val_rmse: 0.9210 - val_smape: 1.9470

Epoch 4/128                                           

287/287 - 5s - 17ms/step - ia: 0.1454 - loss: 1.0334 - mae: 0.8435 - rmse: 1.0156 - smape: 1.6574 - val_ia: 0.2142 - val_loss: 0.8599 - val_mae: 0.7719 - val_rmse: 0.9202 - val_smape: 1.9424

Epoch 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                      

2292/2292 - 71s - 31ms/step - ia: 0.7812 - loss: 0.1874 - mae: 0.3200 - rmse: 0.4075 - smape: 0.6825 - val_ia: 0.3404 - val_loss: 0.1990 - val_mae: 0.3515 - val_rmse: 0.3879 - val_smape: 0.7344

Epoch 2/128                                                                      

2292/2292 - 51s - 22ms/step - ia: 0.8487 - loss: 0.0971 - mae: 0.2283 - rmse: 0.3012 - smape: 0.5535 - val_ia: 0.3901 - val_loss: 0.1986 - val_mae: 0.3252 - val_rmse: 0.3576 - val_smape: 0.6808

Epoch 3/128                                                                      

2292/2292 - 48s - 21ms/step - ia: 0.8614 - loss: 0.0830 - mae: 0.2101 - rmse: 0.2779 - smape: 0.5236 - val_ia: 0.3435 - val_loss: 0.2175 - val_mae: 0.3567 - val_rmse: 0.3938 - val_smape: 0.7254

Epoch 4/128                                                                      

2292/2292 - 52s - 23ms/step - ia: 0.8695 - loss: 0.0751 - mae: 0.1984 - rmse: 0.264

...

Epoch 1/128                                                                         

2292/2292 - 32s - 14ms/step - ia: 0.3401 - loss: 0.8885 - mae: 0.7794 - rmse: 0.9288 - smape: 1.3926 - val_ia: 0.1809 - val_loss: 1.2966 - val_mae: 0.9441 - val_rmse: 0.9757 - val_smape: 1.3513

Epoch 2/128                                                                         

2292/2292 - 41s - 18ms/step - ia: 0.5591 - loss: 0.5574 - mae: 0.6022 - rmse: 0.7358 - smape: 1.0308 - val_ia: 0.1780 - val_loss: 1.2883 - val_mae: 0.9408 - val_rmse: 0.9728 - val_smape: 1.3629

Epoch 3/128                                                                         

2292/2292 - 26s - 11ms/step - ia: 0.6129 - loss: 0.4627 - mae: 0.5433 - rmse: 0.6697 - smape: 0.9347 - val_ia: 0.1760 - val_loss: 1.3403 - val_mae: 0.9682 - val_rmse: 1.0001 - val_smape: 1.3798

Epoch 4/128                                                                         

2292/2292 - 40s - 18ms/step - ia: 0.6485 - loss: 0.4011 - mae: 0.5055 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

4584/4584 - 36s - 8ms/step - ia: 0.2444 - loss: 1.5229 - mae: 0.9487 - rmse: 1.1765 - smape: 1.6040 - val_ia: 0.1475 - val_loss: 0.8240 - val_mae: 0.7306 - val_rmse: 0.7470 - val_smape: 1.5078

Epoch 2/128                                                                          

4584/4584 - 28s - 6ms/step - ia: 0.2439 - loss: 1.4007 - mae: 0.9249 - rmse: 1.1376 - smape: 1.6173 - val_ia: 0.1465 - val_loss: 0.8164 - val_mae: 0.7298 - val_rmse: 0.7460 - val_smape: 1.5342

Epoch 3/128                                                                          

4584/4584 - 29s - 6ms/step - ia: 0.2400 - loss: 1.3205 - mae: 0.9070 - rmse: 1.1091 - smape: 1.6279 - val_ia: 0.1460 - val_loss: 0.8112 - val_mae: 0.7301 - val_rmse: 0.7461 - val_smape: 1.5613

Epoch 4/128                                                                          

4584/4584 - 26s - 6ms/step - ia: 0.2406 - loss: 1.2587 - mae: 0.8939 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                           

287/287 - 10s - 34ms/step - ia: 0.3594 - loss: 0.8128 - mae: 0.7490 - rmse: 0.8982 - smape: 1.3576 - val_ia: 0.4173 - val_loss: 1.0892 - val_mae: 0.8492 - val_rmse: 1.0206 - val_smape: 1.3101

Epoch 2/128                                                                           

287/287 - 10s - 34ms/step - ia: 0.5502 - loss: 0.5871 - mae: 0.6191 - rmse: 0.7647 - smape: 1.0801 - val_ia: 0.4122 - val_loss: 1.4557 - val_mae: 0.9989 - val_rmse: 1.1724 - val_smape: 1.3531

Epoch 3/128                                                                           

287/287 - 10s - 36ms/step - ia: 0.6107 - loss: 0.4892 - mae: 0.5600 - rmse: 0.6978 - smape: 0.9779 - val_ia: 0.4247 - val_loss: 1.4142 - val_mae: 0.9831 - val_rmse: 1.1502 - val_smape: 1.3236

Epoch 4/128                                                                           

287/287 - 4s - 15ms/step - ia: 0.6518 - loss: 0.4219 - mae: 0.5179 - 

In [22]:
print(best)

{'activation': 1, 'batch': 4, 'dropout': 0.2, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}
